# 🚀 EVALUASI EKSTRINSIK: DOWNSTREAM NER TASK (INDOBERT)
## Pembuktian Pengaruh Perluasan Dataset Terhadap Performa Model Deep Learning

Notebook ini mengeksekusi **Evaluasi Ekstrinsik** untuk Bab IV Skripsi:
Membandingkan performa model Transformer **IndoBERT (`indobenchmark/indobert-base-p1`)**:
1. **Model A (Baseline)**: Dilatih pada dataset awal IndQNER 1.0 (1.050 ayat, 2.095 entitas).
2. **Model B (Extended)**: Dilatih pada dataset hasil perluasan (5.353 ayat, 5.649 entitas).
3. **Evaluasi Bersama pada Unseen Test Set**: Kedua model diuji secara independen pada `test.txt` (176 ayat, 237 entitas) yang **100% diisolasi (0% data leakage)**.

> **Panduan GPU**: Pastikan runtime Colab diatur ke **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

---
### 1. Instalasi Dependensi GPU T4

In [1]:
!pip install -q transformers datasets accelerate seqeval evaluate tabulate pandas

import torch
print("=" * 60)
print(f"CUDA Tersedia       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name         : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM Tersedia : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 60)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
CUDA Tersedia       : True
Device Name         : Tesla T4
Total VRAM Tersedia : 15.64 GB


---
### 2. Unggah Berkas Arsip Dataset (`downstream_data.zip`)

In [2]:
import os, zipfile, json
from google.colab import files

if not os.path.exists("downstream_data.zip"):
    print("Silakan unggah berkas 'downstream_data.zip' dari folder 'data/downstream/':")
    uploaded = files.upload()

with zipfile.ZipFile("downstream_data.zip", "r") as zip_ref:
    zip_ref.extractall("downstream_data")

with open("downstream_data/downstream_dataset_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)

print("=" * 60)
print("PARTISI DATASET DOWNSTREAM BERHASIL DIMUAT (0% DATA LEAKAGE):")
print("=" * 60)
for k, v in summary.items():
    print(f"• {k:<15}: {v['verses']:>5d} ayat | {v['tokens']:>7d} token | {v['entities']:>5d} entitas")

Silakan unggah berkas 'downstream_data.zip' dari folder 'data/downstream/':


Saving downstream_data.zip to downstream_data.zip
PARTISI DATASET DOWNSTREAM BERHASIL DIMUAT (0% DATA LEAKAGE):
• baseline_train :  1050 ayat |   49076 token |  2095 entitas
• extended_train :  5353 ayat |  162262 token |  5649 entitas
• dev            :   161 ayat |    6895 token |   231 entitas
• test           :   176 ayat |    6056 token |   237 entitas


---
### 3. Pemuat Format CoNLL BIO & Pembuat Label Map

In [3]:
def load_conll_file(filepath):
    sentences = []
    with open(filepath, "r", encoding="utf-8") as f:
        blocks = f.read().strip().split("\n\n")
        for b in blocks:
            words, tags = [], []
            for line in b.strip().split("\n"):
                parts = line.split("\t")
                if len(parts) >= 2:
                    words.append(parts[0])
                    tags.append(parts[1])
            if words:
                sentences.append({"tokens": words, "ner_tags": tags})
    return sentences

raw_baseline_train = load_conll_file("downstream_data/baseline_train.txt")
raw_extended_train = load_conll_file("downstream_data/extended_train.txt")
raw_dev = load_conll_file("downstream_data/dev.txt")
raw_test = load_conll_file("downstream_data/test.txt")

# Ekstrak seluruh label unik dari seluruh dataset
all_tags = set()
for s in raw_extended_train + raw_dev + raw_test:
    all_tags.update(s["ner_tags"])

label_list = sorted(list(all_tags))
# Pastikan O ada di awal
if "O" in label_list:
    label_list.remove("O")
    label_list = ["O"] + label_list

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}

print(f"✅ Teridentifikasi {len(label_list)} tag unik CoNLL BIO:")
print(label_list)

✅ Teridentifikasi 31 tag unik CoNLL BIO:
['O', 'B-AfterlifeLocation', 'B-Allah', 'B-Angel', 'B-Artifact', 'B-AstronomicalBody', 'B-Color', 'B-Event', 'B-FalseDeity', 'B-Food', 'B-GeographicalLocation', 'B-HolyBook', 'B-Language', 'B-Messenger', 'B-Person', 'B-Prophet', 'B-Religion', 'B-TheBookOfAllah', 'B-Throne', 'I-AfterlifeLocation', 'I-Allah', 'I-Artifact', 'I-AstronomicalBody', 'I-Color', 'I-Event', 'I-GeographicalLocation', 'I-Language', 'I-Messenger', 'I-Person', 'I-Religion', 'I-TheBookOfAllah']


---
### 4. Tokenisasi WordPiece & Penyelarasan Sub-word Label IndoBERT

In [4]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_CHECKPOINT = "indobenchmark/indobert-base-p1"
print(f"⏳ Memuat tokenizer IndoBERT ({MODEL_CHECKPOINT})...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True, max_length=256)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Token spesial [CLS], [SEP], [PAD]
            elif word_idx != previous_word_idx:
                label_ids.append(label2id.get(label[word_idx], 0))
            else:
                # Sub-word token lanjutan: beri -100 agar loss diabaikan (standar CoNLL)
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Konversi ke HuggingFace Dataset
ds_baseline_train = Dataset.from_list(raw_baseline_train).map(tokenize_and_align_labels, batched=True)
ds_extended_train = Dataset.from_list(raw_extended_train).map(tokenize_and_align_labels, batched=True)
ds_dev = Dataset.from_list(raw_dev).map(tokenize_and_align_labels, batched=True)
ds_test = Dataset.from_list(raw_test).map(tokenize_and_align_labels, batched=True)

print("✅ Seluruh dataset berhasil ditokenisasi dan diselaraskan dengan sub-word labels!")

⏳ Memuat tokenizer IndoBERT (indobenchmark/indobert-base-p1)...


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/1050 [00:00<?, ? examples/s]

Map:   0%|          | 0/5353 [00:00<?, ? examples/s]

Map:   0%|          | 0/161 [00:00<?, ? examples/s]

Map:   0%|          | 0/176 [00:00<?, ? examples/s]

✅ Seluruh dataset berhasil ditokenisasi dan diselaraskan dengan sub-word labels!


---
### 5. Definisi Metrik Evaluasi Resmi CoNLL (`seqeval`)

In [5]:
import numpy as np
from seqeval.metrics import f1_score, precision_score, recall_score, classification_report
from seqeval.scheme import IOB2

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p_idx] for (p_idx, l_idx) in zip(prediction, label) if l_idx != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l_idx] for (p_idx, l_idx) in zip(prediction, label) if l_idx != -100]
        for prediction, label in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions, mode='strict', scheme=IOB2),
        "recall": recall_score(true_labels, true_predictions, mode='strict', scheme=IOB2),
        "f1": f1_score(true_labels, true_predictions, mode='strict', scheme=IOB2),
    }

---
### 6. Pelatihan Model A: Baseline IndoBERT (Dilatih pada 1.050 Ayat)

In [8]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

print("\n🏋️ Memulai Pelatihan Model A (Baseline IndoBERT - IndQNER 1.0)...")
model_baseline = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

args_baseline = TrainingArguments(
    output_dir="output_baseline_indobert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=20,
    seed=42
)

trainer_baseline = Trainer(
    model=model_baseline,
    args=args_baseline,
    train_dataset=ds_baseline_train,
    eval_dataset=ds_dev,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_baseline.train()
print("✅ Pelatihan Model A (Baseline) selesai!")


🏋️ Memulai Pelatihan Model A (Baseline IndoBERT - IndQNER 1.0)...


[transformers] You passed `num_labels=31` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.051827,0.038576,0.871901,0.913420,0.892178
2,0.016836,0.030580,0.888430,0.930736,0.909091
3,0.012694,0.033539,0.895833,0.930736,0.912951


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Pelatihan Model A (Baseline) selesai!


---
### 7. Pelatihan Model B: Extended IndoBERT (Dilatih pada 5.353 Ayat)

In [10]:
print("\n🏋️ Memulai Pelatihan Model B (Extended IndoBERT - Korpus 114 Surah)...")
model_extended = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

args_extended = TrainingArguments(
    output_dir="output_extended_indobert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    seed=42
)

trainer_extended = Trainer(
    model=model_extended,
    args=args_extended,
    train_dataset=ds_extended_train,
    eval_dataset=ds_dev,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer_extended.train()
print("✅ Pelatihan Model B (Extended) selesai!")


🏋️ Memulai Pelatihan Model B (Extended IndoBERT - Korpus 114 Surah)...


[transformers] You passed `num_labels=31` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.027646,0.033996,0.894515,0.917749,0.905983
2,0.016255,0.026193,0.927660,0.943723,0.935622
3,0.006938,0.028574,0.922078,0.922078,0.922078


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Pelatihan Model B (Extended) selesai!


---
### 8. Evaluasi Akhir Independen pada Unseen Test Set (`test.txt` - 176 Ayat)

In [11]:
print("\n🔍 Mengevaluasi Model A (Baseline) pada Unseen Test Set...")
preds_a_output = trainer_baseline.predict(ds_test)
metrics_a = preds_a_output.metrics

print("\n🔍 Mengevaluasi Model B (Extended) pada Unseen Test Set...")
preds_b_output = trainer_extended.predict(ds_test)
metrics_b = preds_b_output.metrics

# Ekstrak label list untuk classification report
def get_true_and_pred_labels(preds_output, dataset):
    raw_preds = np.argmax(preds_output.predictions, axis=2)
    labels = preds_output.label_ids
    t_preds = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(raw_preds, labels)
    ]
    t_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(raw_preds, labels)
    ]
    return t_labels, t_preds

y_true_a, y_pred_a = get_true_and_pred_labels(preds_a_output, ds_test)
y_true_b, y_pred_b = get_true_and_pred_labels(preds_b_output, ds_test)


🔍 Mengevaluasi Model A (Baseline) pada Unseen Test Set...



🔍 Mengevaluasi Model B (Extended) pada Unseen Test Set...


---
### 9. Tabel Komparasi Pembuktian Hipotesis Ilmiah (Bab IV Skripsi)

In [12]:
from tabulate import tabulate

prec_a, rec_a, f1_a = metrics_a["test_precision"], metrics_a["test_recall"], metrics_a["test_f1"]
prec_b, rec_b, f1_b = metrics_b["test_precision"], metrics_b["test_recall"], metrics_b["test_f1"]

delta_prec = (prec_b - prec_a) * 100
delta_rec = (rec_b - rec_a) * 100
delta_f1 = (f1_b - f1_a) * 100

table_downstream = [
    ["Jumlah Data Latih (Ayat)", f"{len(raw_baseline_train)} ayat", f"{len(raw_extended_train)} ayat", f"+{len(raw_extended_train)-len(raw_baseline_train)} ayat"],
    ["Jumlah Entitas Latih", f"{summary['baseline_train']['entities']} entitas", f"{summary['extended_train']['entities']} entitas", f"+{summary['extended_train']['entities']-summary['baseline_train']['entities']} entitas"],
    ["Strict Precision", f"{prec_a*100:.2f}%", f"{prec_b*100:.2f}%", f"{delta_prec:+.2f}%"],
    ["Strict Recall", f"{rec_a*100:.2f}%", f"{rec_b*100:.2f}%", f"{delta_rec:+.2f}%"],
    ["Strict F1-Score", f"{f1_a*100:.2f}%", f"{f1_b*100:.2f}%", f"{delta_f1:+.2f}%"]
]

print("=" * 85)
print("HASIL EVALUASI EKSTRINSIK (DOWNSTREAM NER INDOBERT PADA UNSEEN TEST SET)")
print("=" * 85)
print(tabulate(table_downstream, headers=["Parameter / Metrik", "Model A (Baseline IndQNER 1.0)", "Model B (Extended IndQNER)", "Delta (Peningkatan)"], tablefmt="github"))

print("\n" + "=" * 85)
print("DETAIL CLASSIFICATION REPORT MODEL B (EXTENDED INDQNER):")
print("=" * 85)
print(classification_report(y_true_b, y_pred_b, mode='strict', scheme=IOB2, digits=4))

HASIL EVALUASI EKSTRINSIK (DOWNSTREAM NER INDOBERT PADA UNSEEN TEST SET)
| Parameter / Metrik       | Model A (Baseline IndQNER 1.0)   | Model B (Extended IndQNER)   | Delta (Peningkatan)   |
|--------------------------|----------------------------------|------------------------------|-----------------------|
| Jumlah Data Latih (Ayat) | 1050 ayat                        | 5353 ayat                    | +4303 ayat            |
| Jumlah Entitas Latih     | 2095 entitas                     | 5649 entitas                 | +3554 entitas         |
| Strict Precision         | 93.42%                           | 92.86%                       | -0.56%                |
| Strict Recall            | 95.78%                           | 93.25%                       | -2.53%                |
| Strict F1-Score          | 94.58%                           | 93.05%                       | -1.53%                |

DETAIL CLASSIFICATION REPORT MODEL B (EXTENDED INDQNER):
                      precision    r

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


---
### 10. Ekspor Laporan Hasil Downstream (`downstream_evaluation_results.json`)

In [14]:
import json
import numpy as np

# Encoder kustom untuk menangani tipe data NumPy
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

downstream_export = {
    "metadata": {
        "task": "Downstream Named Entity Recognition (Extrinsic Evaluation)",
        "backbone_model": MODEL_CHECKPOINT,
        "hardware": "Google Colab GPU T4",
        "data_summary": summary
    },
    "model_a_baseline": {
        "precision": prec_a,
        "recall": rec_a,
        "f1_score": f1_a,
        "classification_report": classification_report(y_true_a, y_pred_a, mode='strict', scheme=IOB2, output_dict=True)
    },
    "model_b_extended": {
        "precision": prec_b,
        "recall": rec_b,
        "f1_score": f1_b,
        "classification_report": classification_report(y_true_b, y_pred_b, mode='strict', scheme=IOB2, output_dict=True)
    },
    "comparison": {
        "delta_precision": delta_prec,
        "delta_recall": delta_rec,
        "delta_f1_score": delta_f1
    }
}

output_file = "downstream_evaluation_results.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(downstream_export, f, ensure_ascii=False, indent=2, cls=NpEncoder)

print(f"💾 Berkas hasil evaluasi ekstrinsik '{output_file}' berhasil dibuat!")
files.download(output_file)
print("🎉 Berkas telah diunduh otomatis. Siap dimasukkan ke draf Bab IV Skripsi!")

💾 Berkas hasil evaluasi ekstrinsik 'downstream_evaluation_results.json' berhasil dibuat!


/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Berkas telah diunduh otomatis. Siap dimasukkan ke draf Bab IV Skripsi!
